# Prelimenary Blame Detection (PBD)

#### This pipeline serves to do prelimenary localization of blame on translated model data for computationally asssisted labeling of blame. To achieve this goal, the following conceptual steps will be implemented:

- Prepare hypothesis for NLI DEBATE model
- Setup for model
- Read translated sentences
- Pass translated sentences to each hypothesis
- For each hypothesis, evaluate blame based on heuristics of certainty of blame
- 

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("../../src").resolve()))

from PBD import DebateEntailment
from pathlib import Path
import os

In [ ]:

input_path = Path(os.path.join("..",
                      "..",
                      "..",
                      "data_outside_git",
                      "training_data",
                      "translated_train_validation.jsonl"))

output_path = Path(os.path.join("..",
                      "..",
                      "..",
                      "data_outside_git",
                      "training_data",
                        "PBD_and_translated_train_validation.jsonl"))

hyp_template_path = Path(os.path.join(".",
                                      "hypotheis_templates.txt"))

In [ ]:
DE = DebateEntailment(inpath=input_path,
                      outpath=output_path,
                      hyp_templates = hyp_template_path,
                      batch_size=2)

DE.run_hypothesis_entailment()

# Terminal

Can also be run from terminal with arguments. Shoudl automatically detect CUDA

Required:
--input_path_jsonl <path/to/translated/data> 
--output_path_jsonl <path/to/translated/with/PBD>
--hyp_temp_path <path/to/hypothesis_templates.txt>

Optional
--batch_size int (default: 2)





markus copy to terminal:

python PBD.py --input_path_jsonl /work/MarkusLundsfrydJensen#1865/data_outside_git/training_data/translated_train_validation.jsonl --output_path_jsonl /work/MarkusLundsfrydJensen#1865/data_outside_git/training_data/PBD_and_translated_train_validation.jsonl --hyp_temp_path /work/MarkusLundsfrydJensen#1865/Bachelor_project/nbs/2_machine_translation_and_PBD/hypothesis_templates.txt --batch_size 128

# Prelimenary dataset creating


This part serves to create five datasets of differentiated level og agreement between templates. These datasets are to be used to find the most representative account of blame by level of conservatism operationalized by the level og agreement between entailment of blame by hypothesis template.

A validation set will also be made of provided size. The validation set will be balanced. SO if not enough true labels are found, size will be twice that of true labels available. Text which is part of validation set are excluded from the five datasets of different levels of agreement.

A cleanup function has been incoorporated so that no-longer-used entries are deleted to save space

Additionally, this pipeline also provids minimalistic visual and statistical representation of the template-wise differences in amount of entailment of blame found pr template

In [ ]:
import os
from pathlib import Path

from template_manipulation import TemplateManipulation

In [ ]:
#setup paths
inpath = Path(os.path.join("..",
                      "..",
                      "..",
                      "data_outside_git",
                      "training_data",
                        "PBD_and_translated_train_validation.jsonl"))

outpath = Path(os.path.join("..",
                      "..",
                      "..",
                      "data_outside_git",
                      "training_data",
                        "agreement_PBD_and_translated_train_validation.jsonl"))


# make list of keys to delete for datasets of different agreement and validation file

omit_keys = ["translated_text", "n_hyp_entail"]

n_hyp = 5
for i in range(1,n_hyp +1):
    key_name = f"Hyp_{i}_blame"

    omit_keys.append(key_name)

print(omit_keys)

In [ ]:
TM = TemplateManipulation(input_path = inpath,
                            output_path = outpath,
                            validation_size = 500, 
                            clean_keys = omit_keys)

TM.create_datasets()
TM.run_statistics(ylim=5)

The above pipeline can be run from terminal by using the following commands

python template_manipulation.py --input_path <path/to/translated_PBD.jsonl> --output_path <path/to/save_agreement_of_hypothesis.jsonl>

optional arguments:

--ylim int (y limit for plot showing statistics, default 10)
--validation_size int (size of validation size to attempt extraction, default 300)
--clean_keys (repeated arguments for keys to exclude, i.e: --clean_keys Hyp_1_blame Hyp_2_blame Hyp_3_blame)

markus copy terminal:

python template_manipulation.py --input_path /work/MarkusLundsfrydJensen#1865/data_outside_git/training_data/PBD_and_translated_train_validation.jsonl --output_path /work/MarkusLundsfrydJensen#1865/data_outside_git/training_data/agreement_and_PBD_and_translated_train_validation.jsonl --validation_size 500 --ylim 5 --clean_keys translated_text n_hyp_entail Hyp_1_blame Hyp_2_blame Hyp_3_blame Hyp_4_blame Hyp_5_blame

